# La gestion des erreurs

Quand Python rencontre un problème (fichier introuvable, division par zéro, type incorrect...),
il **lève une exception** — un signal d'erreur qui remonte jusqu'à être attrapé
ou jusqu'à faire planter le programme.

Sans gestion d'erreurs → le programme s'arrête brutalement.
Avec gestion d'erreurs → on choisit comment réagir et le programme continue.

```python
try:
    # code qui peut échouer
except TypeError:
    # que faire si ça échoue
else:
    # exécuté seulement si aucune erreur
finally:
    # exécuté TOUJOURS (erreur ou pas)
```

---
## 1. Structure complète : `try / except / else / finally`

| Bloc | Rôle |
|---|---|
| `try` | Le code qui peut échouer |
| `except` | Ce qu'on fait si ça échoue |
| `else` | Exécuté **seulement si aucune erreur** dans le `try` |
| `finally` | Exécuté **toujours** — erreur ou pas (nettoyage) |

In [ ]:
# Cas 1 : le try réussit
try:
    x = int("42")       # conversion qui réussit
    print(f"x = {x}")
except ValueError:
    print("Impossible de convertir en entier")
else:
    print("else : aucune erreur — x est valide")
finally:
    print("finally : toujours exécuté")

In [ ]:
# Cas 2 : le try échoue
try:
    x = int("abc")      # lève ValueError
    print(f"x = {x}")
except ValueError:
    print("Impossible de convertir en entier")  # capturé ici
else:
    print("else : jamais atteint si erreur")
finally:
    print("finally : toujours exécuté")

---
## 2. Les exceptions courantes

Il faut **toujours préciser le type d'exception** — un `except:` nu attrape tout,
même les erreurs système (`KeyboardInterrupt`, `SystemExit`), ce qui masque des bugs réels.

| Exception | Cause typique |
|---|---|
| `ValueError` | Valeur incorrecte — `int("abc")` |
| `TypeError` | Type incompatible — `"a" + 1` |
| `ZeroDivisionError` | Division par zéro |
| `IndexError` | Index hors limites — `liste[999]` |
| `KeyError` | Clé absente dans un dict — `d["inexistante"]` |
| `AttributeError` | Attribut inexistant — `None.upper()` |
| `FileNotFoundError` | Fichier introuvable |
| `NameError` | Variable non définie |
| `RecursionError` | Trop d'appels récursifs |

In [ ]:
# Plusieurs except pour gérer différents cas
def convertir_et_diviser(texte, diviseur):
    try:
        nombre = int(texte)
        resultat = nombre / diviseur
        return resultat
    except ValueError:
        print(f"'{texte}' n'est pas un entier valide")
    except ZeroDivisionError:
        print('Division par zéro impossible')
    except Exception as e:
        # filet de sécurité pour tout ce qu'on n'a pas prévu
        print(f'Erreur inattendue : {type(e).__name__} — {e}')

print(convertir_et_diviser("10", 2))    # 5.0
print(convertir_et_diviser("abc", 2))   # ValueError
print(convertir_et_diviser("10", 0))    # ZeroDivisionError

---
## 3. Récupérer le message d'erreur avec `as`

`except TypeError as e` stocke l'objet exception dans `e`.
On peut afficher le message, le logger, ou inspecter le type.

In [ ]:
try:
    resultat = "100" + 5    # TypeError : str + int impossible
except TypeError as e:
    print(f'Type : {type(e).__name__}')   # TypeError
    print(f'Message : {e}')               # can only concatenate str...

# Attraper plusieurs types dans un seul except
try:
    liste = [1, 2, 3]
    print(liste[10])           # IndexError
except (IndexError, KeyError) as e:
    print(f'Problème d\'accès : {e}')

---
## 4. Lever une exception avec `raise`

`raise` signale volontairement qu'une condition anormale est atteinte.
C'est la façon propre de communiquer une erreur à l'appelant
— pas de `return None`, pas de `return -1`.

`raise` seul (sans argument) re-lève l'exception en cours.
Utile pour logger puis laisser l'erreur remonter.

In [ ]:
def diviser(a, b):
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
        raise TypeError(
            f'Les deux arguments doivent être des nombres, '
            f'reçu : {type(a).__name__} et {type(b).__name__}'
        )
    if b == 0:
        raise ZeroDivisionError('Le diviseur ne peut pas être zéro')
    return a / b

print(diviser(10, 2))     # 5.0

try:
    diviser(10, 0)
except ZeroDivisionError as e:
    print(f'Erreur : {e}')

try:
    diviser("10", 2)
except TypeError as e:
    print(f'Erreur : {e}')

In [ ]:
# raise seul — re-lever après avoir loggé
def lire_config(chemin):
    try:
        with open(chemin) as f:
            return f.read()
    except FileNotFoundError as e:
        print(f'[LOG] Fichier de config introuvable : {chemin}')
        raise   # re-lève la même exception — l'appelant doit la gérer

try:
    lire_config('config_inexistante.json')
except FileNotFoundError:
    print('L\'application ne peut pas démarrer sans config')

---
## 5. Créer ses propres exceptions

Une exception personnalisée est une classe qui hérite de `Exception`.
Elle permet de distinguer tes erreurs métier des erreurs Python standard.

```python
class MonErreur(Exception):
    pass
```

C'est tout ce qu'il faut pour une exception simple.
On peut ajouter des attributs pour transporter plus d'informations.

In [ ]:
# Exceptions simples
class AgeInvalideError(Exception):
    pass

class StockInsuffisantError(Exception):
    pass

# Avec attributs supplémentaires
class PrixNegatifError(Exception):
    def __init__(self, prix, produit):
        self.prix = prix
        self.produit = produit
        super().__init__(f'Prix négatif ({prix}€) pour le produit "{produit}"')

# ── Utilisation ──
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f'L\'âge doit être un entier, reçu : {type(age).__name__}')
    if age < 0 or age > 150:
        raise AgeInvalideError(f'Âge invalide : {age}')
    return age

for valeur in [25, -5, 200, 'trente']:
    try:
        print(f'set_age({valeur!r}) → {set_age(valeur)}')
    except (AgeInvalideError, TypeError) as e:
        print(f'  ✗ {type(e).__name__}: {e}')

In [ ]:
# Exception avec attributs métier
class Stock:
    def __init__(self, produit, quantite, prix):
        if prix < 0:
            raise PrixNegatifError(prix, produit)
        self.produit = produit
        self.quantite = quantite
        self.prix = prix

    def vendre(self, quantite):
        if quantite > self.quantite:
            raise StockInsuffisantError(
                f'Stock insuffisant : demandé {quantite}, disponible {self.quantite}'
            )
        self.quantite -= quantite

try:
    s = Stock("Laptop", 5, 999)
    s.vendre(3)
    print(f'Vente OK — stock restant : {s.quantite}')
    s.vendre(10)   # plus assez
except StockInsuffisantError as e:
    print(f'Vente refusée : {e}')

try:
    Stock("Produit", 10, -50)
except PrixNegatifError as e:
    print(f'Création refusée : {e}')
    print(f'  prix={e.prix}, produit={e.produit}')

---
## 6. Hiérarchie des exceptions

Les exceptions Python forment un arbre d'héritage.
Attraper un parent attrape aussi tous ses enfants.

```
BaseException
└── Exception                    ← attraper ici pour tout ce qui est normal
    ├── ValueError
    ├── TypeError
    ├── ArithmeticError
    │   └── ZeroDivisionError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    └── OSError
        └── FileNotFoundError
```

> **En entretien** : `except BaseException` attrape aussi `KeyboardInterrupt` (Ctrl+C)
> et `SystemExit`. Ne jamais les attraper sauf cas très précis.
> Utilise toujours `except Exception` au maximum.

In [ ]:
# Démonstration de la hiérarchie
erreurs = [
    ValueError("mauvaise valeur"),
    TypeError("mauvais type"),
    ZeroDivisionError("division par zéro"),
    IndexError("index hors limites"),
]

for err in erreurs:
    if isinstance(err, LookupError):
        print(f'{type(err).__name__} → attrapé par LookupError')
    elif isinstance(err, ArithmeticError):
        print(f'{type(err).__name__} → attrapé par ArithmeticError')
    elif isinstance(err, Exception):
        print(f'{type(err).__name__} → attrapé par Exception')

---
## 7. `with` — gestion automatique des ressources

Le bloc `with` garantit que la ressource (fichier, connexion BD...)
sera **fermée même si une exception se produit**.
C'est l'équivalent d'un `try/finally` automatique.

In [ ]:
# Sans with — fragile : si une exception se produit, close() n'est jamais appelé
f = open("test_erreurs.txt", "w")
try:
    f.write("bonjour")
finally:
    f.close()   # on doit le mettre dans finally pour être sûr

# Avec with — propre, sûr, automatique
with open("test_erreurs.txt", "w") as f:
    f.write("bonjour\n")
    f.write("monde\n")
# f.close() appelé automatiquement ici, erreur ou pas

# Lecture
with open("test_erreurs.txt") as f:
    contenu = f.read()
    print(contenu)

In [ ]:
import os

# Pattern robuste pour lire un fichier
def lire_fichier(chemin):
    try:
        with open(chemin, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return None          # signal propre : fichier absent
    except PermissionError:
        raise                # on ne peut rien faire — l'appelant doit le savoir

contenu = lire_fichier("test_erreurs.txt")
if contenu is not None:
    print(f'Lu {len(contenu)} caractères')
else:
    print('Fichier introuvable')

# Nettoyage
os.remove("test_erreurs.txt")

---
## 8. Patterns à connaître pour l'entretien

### Pattern 1 — Validation en entrée de fonction
Valider les arguments dès le début en levant des exceptions descriptives.
C'est la façon propre — pas de `return -1`, pas de `return None`.

In [ ]:
def calculer_moyenne(notes):
    if not isinstance(notes, list):
        raise TypeError(f'Attendu une liste, reçu {type(notes).__name__}')
    if len(notes) == 0:
        raise ValueError('La liste de notes ne peut pas être vide')
    if not all(isinstance(n, (int, float)) for n in notes):
        raise ValueError('Toutes les notes doivent être des nombres')
    if not all(0 <= n <= 20 for n in notes):
        raise ValueError('Les notes doivent être entre 0 et 20')
    return sum(notes) / len(notes)

cas = [
    [15, 12, 18],
    [],
    [15, 'abc', 18],
    [15, 25, 18],
]

for notes in cas:
    try:
        print(f'Moyenne de {notes} : {calculer_moyenne(notes):.2f}')
    except (ValueError, TypeError) as e:
        print(f'  ✗ {type(e).__name__}: {e}')

### Pattern 2 — Boucle de saisie robuste
Utiliser les exceptions pour gérer les entrées utilisateur incorrectes.
Le `break` ne sort que quand la validation passe.

In [ ]:
# Simulation de saisie utilisateur
saisies = ["abc", "-5", "25", "12"]
index = 0

def saisie_simulee(prompt):
    global index
    val = saisies[index]
    index += 1
    print(f'{prompt}{val}')
    return val

def demander_age():
    while True:
        try:
            texte = saisie_simulee('Entrez votre âge : ')
            age = int(texte)           # ValueError si pas un nombre
            if age < 0 or age > 150:
                raise ValueError(f'Âge hors limites : {age}')
            return age
        except ValueError as e:
            print(f'  Valeur invalide ({e}), réessayez')

age = demander_age()
print(f'\nÂge validé : {age}')

### Pattern 3 — Fonctions utilitaires safe
Retourner `None` ou une valeur par défaut au lieu de laisser l'exception remonter.

In [ ]:
def vers_entier(valeur, defaut=0):
    """Convertit en entier, retourne defaut si impossible."""
    try:
        return int(valeur)
    except (ValueError, TypeError):
        return defaut

def acces_dict_securise(d, cle, defaut=None):
    """Accès à un dict sans risque de KeyError."""
    try:
        return d[cle]
    except KeyError:
        return defaut

print(vers_entier("42"))       # 42
print(vers_entier("abc"))      # 0
print(vers_entier(None, -1))   # -1

config = {"host": "localhost", "port": 5432}
print(acces_dict_securise(config, "host"))        # localhost
print(acces_dict_securise(config, "password"))    # None
# Note : d.get('clé', defaut) fait la même chose nativement pour les dicts

---
## Récapitulatif

| Élément | Rôle |
|---|---|
| `try` | Bloc de code qui peut échouer |
| `except TypeError` | Réaction à un type d'erreur précis |
| `except (A, B) as e` | Plusieurs types, message accessible via `e` |
| `else` | Exécuté seulement si **aucune** erreur |
| `finally` | Exécuté **toujours** (nettoyage) |
| `raise TypeError('msg')` | Lever volontairement une exception |
| `raise` seul | Re-lever l'exception courante |
| `class MonErreur(Exception)` | Exception personnalisée |
| `with open(...) as f` | Fermeture automatique garantie |

### Règles à retenir
- Toujours préciser le type dans `except` — jamais `except:` nu
- Lever des exceptions dès l'entrée de fonction si les arguments sont invalides
- Préférer `with` pour toute ressource (fichier, connexion, verrou)
- Créer ses propres exceptions pour les erreurs métier